# Forest cover type classification — final stacking model

This is the report-aligned final four-model Stacking workflow. Run Jupyter from the repository root and place the authorized Kaggle files in `data/local/`.


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "notebooks/final_stacking.ipynb").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from inside the cloned project directory.")
DATA_DIR = PROJECT_ROOT / "data/local"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.neighbors import RadiusNeighborsClassifier
import xgboost as xgb
from sklearn.ensemble import ExtraTreesClassifier

In [ ]:
def add_geo_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # Aspect -> sin / cos
    theta = np.deg2rad(out["Aspect"].astype(float))
    out["Aspect_sin"] = np.sin(theta)
    out["Aspect_cos"] = np.cos(theta)

    # 与水体的欧几里得距离
    out["Euclidean_Distance_To_Hydrology"] = np.sqrt(
        out["Horizontal_Distance_To_Hydrology"]**2 + out["Vertical_Distance_To_Hydrology"]**2
    )

    # Hillshade 统计量
    out["Hillshade_max"] = out[["Hillshade_9am", "Hillshade_Noon", "Hillshade_3pm"]].max(axis=1)
    out["Hillshade_min"] = out[["Hillshade_9am", "Hillshade_Noon", "Hillshade_3pm"]].min(axis=1)
    out["Hillshade_mean"] = out[["Hillshade_9am", "Hillshade_Noon", "Hillshade_3pm"]].mean(axis=1)
    out["Hillshade_range"] = out["Hillshade_max"] - out["Hillshade_min"]

    # Elevation 组合
    out["Elevation_Plus_Vertical_Hydrology"] = out["Elevation"] + out["Vertical_Distance_To_Hydrology"]
    out["Elevation_Minus_Vertical_Hydrology"] = out["Elevation"] - out["Vertical_Distance_To_Hydrology"]
    out["Elevation_Plus_Horizontal_Roadways"] = out["Elevation"] + out["Horizontal_Distance_To_Roadways"]
    out["Elevation_Minus_Horizontal_Roadways"] = out["Elevation"] - out["Horizontal_Distance_To_Roadways"]

    # FirePoints & Roadways
    out["Fire_Points_Plus_Roadways"] = out["Horizontal_Distance_To_Fire_Points"] + out["Horizontal_Distance_To_Roadways"]
    out["Fire_Points_Minus_Roadways"] = out["Horizontal_Distance_To_Fire_Points"] - out["Horizontal_Distance_To_Roadways"]

    # Vertical Hydrology 绝对值
    out["Vertical_Distance_To_Hydrology_Abs"] = out["Vertical_Distance_To_Hydrology"].abs()

    # 平均设施距离
    out["Mean_Amenities_Distance"] = (
        out["Horizontal_Distance_To_Hydrology"] +
        out["Horizontal_Distance_To_Fire_Points"] +
        out["Horizontal_Distance_To_Roadways"]
    ) / 3

    # 距离差值
    out["Hydro_Fire_Diff"] = (out["Horizontal_Distance_To_Hydrology"] - out["Horizontal_Distance_To_Fire_Points"]).abs()
    out["Hydro_Road_Diff"] = (out["Horizontal_Distance_To_Hydrology"] - out["Horizontal_Distance_To_Roadways"]).abs()

    # Hydrology & Fire Points
    out["Hydrology_Plus_Fire_Points"] = out["Horizontal_Distance_To_Hydrology"] + out["Horizontal_Distance_To_Fire_Points"]
    out["Hydrology_Minus_Fire_Points"] = out["Horizontal_Distance_To_Hydrology"] - out["Horizontal_Distance_To_Fire_Points"]
    
    # Hydrology & Roadways
    out["Hydrology_Plus_Roadways"] = out["Horizontal_Distance_To_Hydrology"] + out["Horizontal_Distance_To_Roadways"]
    out["Hydrology_Minus_Roadways"] = out["Horizontal_Distance_To_Hydrology"] - out["Horizontal_Distance_To_Roadways"]

    return out

In [ ]:
# 1. 读取数据
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test-full.csv")

train = add_geo_features(train)
test = add_geo_features(test)

X = train.drop(columns=["Cover_Type", "Id"])
y = train["Cover_Type"]

# 2. 划分数据集
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.001, random_state=42, stratify=y
)


In [ ]:
# 3. 定义基学习器
rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    random_state=8,
    max_features=0.3,
    min_samples_leaf=1,
    min_samples_split=2,
    class_weight="balanced",
    n_jobs=-1
)

rnc = Pipeline([
    ("scaler", StandardScaler()),
    ("rnc", RadiusNeighborsClassifier(
        radius=9,
        weights="distance",
        metric="manhattan",
        outlier_label="most_frequent",
        n_jobs=-1
    ))
])

xgb_clf = xgb.XGBClassifier(
    n_estimators=800,
    max_depth=9,
    learning_rate=0.1,
    subsample=1,
    colsample_bytree=1,
    min_child_weight=1,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

et = ExtraTreesClassifier(
    n_estimators=800,
    max_depth=None,
    # min_samples_leaf=1,
    # min_samples_split=2,
    max_features="sqrt",  # 可调参数
    random_state=42,
    n_jobs=-1
)

In [ ]:
# 4. 定义 Stacking 集成（增加 ExtraTrees）
stacking_clf = StackingClassifier(
    estimators=[
        ("rf", rf),
        ("rnc", rnc),
        ("xgb", xgb_clf),
        ("et", et)  # ✅ 新增 ExtraTrees
    ],
    final_estimator=xgb.XGBClassifier(
        n_estimators=500, 
        max_depth=7,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1
    ),
    passthrough=True,
    cv=5,
    n_jobs=-1
)

In [ ]:
# 5. 训练
stacking_clf.fit(X_train, y_train)

In [ ]:
# 6. 验证集预测
y_val_pred = stacking_clf.predict(X_val)
print("✅ Stacking Accuracy:", accuracy_score(y_val, y_val_pred))
print("\n📊 Classification Report:\n", classification_report(y_val, y_val_pred))
print("\n🔎 Confusion Matrix:\n", confusion_matrix(y_val, y_val_pred))

In [ ]:
# 7. 测试集预测
X_test = test[X_train.columns]
test_pred = stacking_clf.predict(X_test)

submission = pd.DataFrame({
    "Id": test["Id"].astype(int),
    "Cover_Type": test_pred.astype(int)
})
submission.to_csv(OUTPUT_DIR / "submission_stacking_4models.csv", index=False)
print("📁 已生成 submission_stacking_4models.csv")

In [ ]:
# random state = 42: 0.83966

In [ ]:
# random state = 66: 0.838

In [ ]:
# random state = 89: 0.836